In [ ]:
# Set the project root
import os

os.chdir("..")
os.getcwd()

In [ ]:
# Imports dependencies
from datetime import date
import pandas as pd

from src.data_models.request import EnergyDataRequest

# 1. Download data

In [ ]:
api_request = EnergyDataRequest(
    start=date(2026, 1, 1),
    end="now",
)

df = api_request.download_data()

In [ ]:
df = pd.read_parquet("data/raw/DayAheadPrices.parquet")

# 2. Inspect the data

In [ ]:
import pandas as pd
from IPython.display import display, Markdown

In [ ]:
def preview_data(df: pd.DataFrame) -> None:
    """Display a Markdown overview of a time-series DataFrame."""

    # Basic information
    rows, columns = df.shape

    # Missing values
    missing = df.isna().sum()

    # Duplicate observations
    duplicates = df.duplicated(["TimeDK", "PriceArea"]).sum()

    # Observation frequency
    timestamps = pd.to_datetime(df["TimeDK"]).sort_values()
    frequency = timestamps.diff().value_counts()

    # Price extremes
    highest = df.nlargest(5, "DayAheadPriceEUR")[
        ["TimeDK", "PriceArea", "DayAheadPriceEUR"]
    ]
    lowest = df.nsmallest(5, "DayAheadPriceEUR")[
        ["TimeDK", "PriceArea", "DayAheadPriceEUR"]
    ]

    # Descriptive statistics
    statistics = df["DayAheadPriceEUR"].describe()

    # Build Markdown output
    overview = f"""
## Dataset overview

| Metric | Value |
|---|---:|
| Number of observations | {rows:,} |
| Number of columns | {columns} |
| Missing values | {missing.sum():,} |
| Duplicate observations | {duplicates:,} |
| Unique price areas | {df["PriceArea"].nunique()} |
| Start date | {timestamps.min()} |
| End date | {timestamps.max()} |

## Column information

| Column | Data type | Missing values |
|---|---|---:|
"""

    for column in df.columns:
        overview += (
            f"| {column} | {df[column].dtype} | "
            f"{missing[column]:,} |\\n"
        )

    overview += f"""
## Price statistics

| Statistic | EUR/MWh |
|---|---:|
"""

    for statistic, value in statistics.items():
        overview += f"| {statistic.capitalize()} | {value:,.2f} |\\n"

    overview += """
## Observation frequency

| Time difference | Count |
|---|---:|
"""

    for interval, count in frequency.items():
        overview += f"| {interval} | {count:,} |\\n"

    overview += """
## Highest prices

| TimeDK | Price area | Price (EUR/MWh) |
|---|---|---:|
"""

    for _, row in highest.iterrows():
        overview += (
            f"| {row['TimeDK']} | {row['PriceArea']} | "
            f"{row['DayAheadPriceEUR']:,.2f} |\\n"
        )

    overview += """
## Lowest prices

| TimeDK | Price area | Price (EUR/MWh) |
|---|---|---:|
"""

    for _, row in lowest.iterrows():
        overview += (
            f"| {row['TimeDK']} | {row['PriceArea']} | "
            f"{row['DayAheadPriceEUR']:,.2f} |\\n"
        )

    display(Markdown(overview))

In [ ]:
preview_data(df)

In [ ]:
df.info() 

In [ ]:
df.head()

In [ ]:
df.isna().sum()

In [ ]:
df.duplicated(["TimeDK", "PriceArea"]).sum()

In [ ]:
df["DayAheadPriceEUR"].nlargest(10)
df["DayAheadPriceEUR"].nsmallest(10)

In [ ]:
from src.clients.postgre import PostgresClient

In [ ]:
db = PostgresClient()

In [ ]:
processed = db.insert_day_ahead_prices(df)

print(f"Processed {processed:,} observations.")

In [ ]:
db.close()

In [ ]:
query = """
SELECT
    price_area,
    AVG(price_eur_mwh) AS average_price,
    MIN(price_eur_mwh) AS minimum_price,
    MAX(price_eur_mwh) AS maximum_price
FROM day_ahead_prices
GROUP BY price_area;
"""

result = pd.read_sql(query, db.engine)
display(result)